
# EMT DC SSN Pi-Line Source-Step

This notebook runs and evaluates the compiled DPsim example:

```text
EMT_DC_SSN_PiLine_SourceStep
```

The simulated network is:

```text
20 kV → 22 kV ideal DC source step at 0.1 s
    │
0.2 Ω feeder
    │
sending node
    │
DC SSN Pi-line: R = 0.5 Ω, L = 20 mH, C = 2 mF
    │
receiving node
    │
20 Ω resistive load
    │
ground
```


In [ ]:
from __future__ import annotations

import math
import os
import re
import subprocess
from pathlib import Path
from typing import Iterable

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 160)

print("Python environment ready.")

## 1. Repository and simulation configuration

In [ ]:
# Adjust this path if your DPsim repository is elsewhere.
REPO_ROOT = Path.home() / "dpsim"
BUILD_DIR = REPO_ROOT / "build"

TARGET_NAME = "EMT_DC_SSN_PiLine_SourceStep"
SIM_NAME = TARGET_NAME

TIME_STEP = 1e-5
FINAL_TIME = 0.5
STEP_TIME = 0.1

INITIAL_SOURCE_VOLTAGE = 20e3
STEPPED_SOURCE_VOLTAGE = 22e3
FEEDER_RESISTANCE = 0.2
LINE_RESISTANCE = 0.5
LINE_INDUCTANCE = 20e-3
TOTAL_LINE_CAPACITANCE = 2e-3
LOAD_RESISTANCE = 20.0

assert REPO_ROOT.exists(), f"DPsim repository not found: {REPO_ROOT}"
assert BUILD_DIR.exists(), f"DPsim build directory not found: {BUILD_DIR}"

print(f"Repository: {REPO_ROOT}")
print(f"Build dir:  {BUILD_DIR}")

## 2. Analytical DC operating points

In [ ]:
def dc_operating_point(source_voltage: float) -> dict[str, float]:
    total_series_resistance = FEEDER_RESISTANCE + LINE_RESISTANCE + LOAD_RESISTANCE
    current = source_voltage / total_series_resistance
    sending_voltage = source_voltage - FEEDER_RESISTANCE * current
    receiving_voltage = LOAD_RESISTANCE * current
    line_drop = LINE_RESISTANCE * current

    return {
        "source_voltage": source_voltage,
        "current": current,
        "sending_voltage": sending_voltage,
        "receiving_voltage": receiving_voltage,
        "line_drop": line_drop,
    }


initial_expected = dc_operating_point(INITIAL_SOURCE_VOLTAGE)
final_expected = dc_operating_point(STEPPED_SOURCE_VOLTAGE)

expected_df = pd.DataFrame(
    [initial_expected, final_expected],
    index=["pre-step", "post-step"],
)
expected_df

## 3. Build the DPsim target

In [ ]:
BUILD_TARGET = True

if BUILD_TARGET:
    build_command = [
        "cmake",
        "--build",
        str(BUILD_DIR),
        "--target",
        TARGET_NAME,
        "--parallel",
        "2",
    ]

    print("Running:")
    print(" ".join(build_command))

    build_result = subprocess.run(
        build_command,
        cwd=REPO_ROOT,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
    )

    print(build_result.stdout)

    if build_result.returncode != 0:
        raise RuntimeError(f"Build failed with exit code {build_result.returncode}.")
else:
    print("Build skipped.")

## 4. Locate and run the executable

In [ ]:
def find_executable(build_dir: Path, target_name: str) -> Path:
    candidates = [
        path
        for path in build_dir.rglob(target_name)
        if path.is_file() and os.access(path, os.X_OK)
    ]

    if not candidates:
        raise FileNotFoundError(
            f"No executable named {target_name!r} found below {build_dir}."
        )

    # Prefer the normal DPsim C++ example location when present.
    candidates.sort(
        key=lambda path: (
            "dpsim/examples/cxx" not in path.as_posix(),
            len(path.parts),
        )
    )
    return candidates[0]


executable = find_executable(BUILD_DIR, TARGET_NAME)
print(f"Executable: {executable}")

In [ ]:
run_result = subprocess.run(
    [str(executable)],
    cwd=REPO_ROOT,
    text=True,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
)

print(run_result.stdout)

if run_result.returncode != 0:
    raise RuntimeError(f"Simulation failed with exit code {run_result.returncode}.")

## 5. Locate and read the DPsim CSV

In [ ]:
def find_latest_csv(repo_root: Path, simulation_name: str) -> Path:
    preferred_root = repo_root / "logs" / simulation_name

    candidates: list[Path] = []
    if preferred_root.exists():
        candidates.extend(preferred_root.rglob("*.csv"))

    if not candidates:
        logs_root = repo_root / "logs"
        if logs_root.exists():
            candidates.extend(
                path
                for path in logs_root.rglob("*.csv")
                if simulation_name.lower() in path.as_posix().lower()
            )

    if not candidates:
        raise FileNotFoundError(
            f"No CSV for {simulation_name} found below {repo_root / 'logs'}."
        )

    return max(candidates, key=lambda path: path.stat().st_mtime)


csv_path = find_latest_csv(REPO_ROOT, SIM_NAME)
print(f"CSV: {csv_path}")

In [ ]:
def read_dpsim_csv(path: Path) -> pd.DataFrame:
    # DPsim logs may use comma, semicolon, or tab delimiters depending on build
    # and logger configuration.
    dataframe = pd.read_csv(path, sep=None, engine="python")

    # Convert columns that contain numeric strings.
    for column in dataframe.columns:
        if dataframe[column].dtype == object:
            converted = pd.to_numeric(dataframe[column], errors="coerce")
            if converted.notna().sum() >= max(1, int(0.95 * len(converted))):
                dataframe[column] = converted

    return dataframe


df = read_dpsim_csv(csv_path)

print(f"Rows:    {len(df)}")
print(f"Columns: {len(df.columns)}")
display(df.head())

In [ ]:
print("Logged columns:")
for column in df.columns:
    print(f"  {column}")

## 6. Resolve time and signal columns

In [ ]:
def normalized_name(name: str) -> str:
    return re.sub(r"[^a-z0-9]+", "_", name.lower()).strip("_")


normalized_columns = {normalized_name(column): column for column in df.columns}


def find_column(
    exact_names: Iterable[str] = (),
    contains_all: Iterable[str] = (),
    contains_any: Iterable[str] = (),
) -> str | None:
    exact_normalized = {normalized_name(name) for name in exact_names}
    all_tokens = tuple(normalized_name(token) for token in contains_all)
    any_tokens = tuple(normalized_name(token) for token in contains_any)

    for normalized, original in normalized_columns.items():
        if normalized in exact_normalized:
            return original

    for normalized, original in normalized_columns.items():
        if all(token in normalized for token in all_tokens):
            if not any_tokens or any(token in normalized for token in any_tokens):
                return original

    return None


time_column = find_column(
    exact_names=("time", "t"),
    contains_all=("time",),
)

if time_column is None:
    raise KeyError("Could not identify the time column.")

signals = {
    "source_node_voltage": find_column(
        exact_names=("v_source_node",),
        contains_all=("source", "node"),
        contains_any=("voltage", "v"),
    ),
    "sending_node_voltage": find_column(
        exact_names=("v_sending_node",),
        contains_all=("sending", "node"),
        contains_any=("voltage", "v"),
    ),
    "receiving_node_voltage": find_column(
        exact_names=("v_receiving_node",),
        contains_all=("receiving", "node"),
        contains_any=("voltage", "v"),
    ),
    "source_current": find_column(
        exact_names=("i_source_intf",),
        contains_all=("source",),
        contains_any=("current", "i_intf"),
    ),
    "feeder_current": find_column(
        exact_names=("i_feeder_intf",),
        contains_all=("feeder",),
        contains_any=("current", "i_intf"),
    ),
    "line_current": find_column(
        exact_names=("i_line_intf",),
        contains_all=("line",),
        contains_any=("current", "i_intf"),
    ),
    "load_current": find_column(
        exact_names=("i_load_intf",),
        contains_all=("load",),
        contains_any=("current", "i_intf"),
    ),
}

print(f"Time column: {time_column}")
print("\nResolved signals:")
for name, column in signals.items():
    print(f"  {name:24s}: {column}")

## 7. Basic data-quality checks

In [ ]:
numeric_df = df.select_dtypes(include=[np.number])

if numeric_df.empty:
    raise RuntimeError("The CSV contains no numeric columns.")

finite_mask = np.isfinite(numeric_df.to_numpy())
print(f"All numeric values finite: {finite_mask.all()}")

if not finite_mask.all():
    invalid_locations = np.argwhere(~finite_mask)
    first_row, first_column = invalid_locations[0]
    raise RuntimeError(
        "Non-finite data detected at "
        f"row {first_row}, column {numeric_df.columns[first_column]!r}."
    )

time = df[time_column].to_numpy(dtype=float)

print(f"Time range: {time.min():.6g} s to {time.max():.6g} s")
print(f"Median logged time step: {np.median(np.diff(time)):.6g} s")

## 8. Node-voltage plots

In [ ]:
voltage_series = [
    ("Source node", signals["source_node_voltage"]),
    ("Sending node", signals["sending_node_voltage"]),
    ("Receiving node", signals["receiving_node_voltage"]),
]
voltage_series = [
    (label, column) for label, column in voltage_series if column is not None
]

fig, ax = plt.subplots(figsize=(12, 5))
for label, column in voltage_series:
    ax.plot(time, df[column], label=label)

ax.axvline(STEP_TIME, linestyle="--", label=f"source step at {STEP_TIME:g} s")
ax.set_xlabel("Time [s]")
ax.set_ylabel("Voltage [V]")
ax.set_title("DC node voltages")
ax.grid(True)
ax.legend()
fig.tight_layout()
plt.show()

## 9. Branch-current plots

In [ ]:
current_series = [
    ("Source", signals["source_current"]),
    ("Feeder", signals["feeder_current"]),
    ("Pi-line", signals["line_current"]),
    ("Load", signals["load_current"]),
]
current_series = [
    (label, column) for label, column in current_series if column is not None
]

fig, ax = plt.subplots(figsize=(12, 5))
for label, column in current_series:
    ax.plot(time, df[column], label=label)

ax.axvline(STEP_TIME, linestyle="--", label=f"source step at {STEP_TIME:g} s")
ax.set_xlabel("Time [s]")
ax.set_ylabel("Current [A]")
ax.set_title("DC branch currents")
ax.grid(True)
ax.legend()
fig.tight_layout()
plt.show()

## 10. Zoom around the source step

In [ ]:
zoom_start = STEP_TIME - 0.02
zoom_end = STEP_TIME + 0.12
zoom_mask = (time >= zoom_start) & (time <= zoom_end)

fig, ax = plt.subplots(figsize=(12, 5))
for label, column in voltage_series:
    ax.plot(time[zoom_mask], df.loc[zoom_mask, column], label=label)

ax.axvline(STEP_TIME, linestyle="--", label=f"source step at {STEP_TIME:g} s")
ax.set_xlabel("Time [s]")
ax.set_ylabel("Voltage [V]")
ax.set_title("DC node voltages around the source step")
ax.grid(True)
ax.legend()
fig.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
for label, column in current_series:
    ax.plot(time[zoom_mask], df.loc[zoom_mask, column], label=label)

ax.axvline(STEP_TIME, linestyle="--", label=f"source step at {STEP_TIME:g} s")
ax.set_xlabel("Time [s]")
ax.set_ylabel("Current [A]")
ax.set_title("DC branch currents around the source step")
ax.grid(True)
ax.legend()
fig.tight_layout()
plt.show()

## 11. Compare simulated and analytical operating points

In [ ]:
def window_mean(column: str | None, start: float, end: float) -> float:
    if column is None:
        return math.nan
    mask = (time >= start) & (time <= end)
    if not mask.any():
        raise ValueError(f"No samples in window [{start}, {end}].")
    return float(df.loc[mask, column].mean())


pre_window = (STEP_TIME - 0.02, STEP_TIME - 0.002)
post_window = (FINAL_TIME - 0.02, FINAL_TIME)

comparison = pd.DataFrame(
    {
        "analytical_pre": {
            "sending_voltage": initial_expected["sending_voltage"],
            "receiving_voltage": initial_expected["receiving_voltage"],
            "line_current": initial_expected["current"],
        },
        "simulated_pre": {
            "sending_voltage": window_mean(
                signals["sending_node_voltage"], *pre_window
            ),
            "receiving_voltage": window_mean(
                signals["receiving_node_voltage"], *pre_window
            ),
            "line_current": window_mean(signals["line_current"], *pre_window),
        },
        "analytical_post": {
            "sending_voltage": final_expected["sending_voltage"],
            "receiving_voltage": final_expected["receiving_voltage"],
            "line_current": final_expected["current"],
        },
        "simulated_post": {
            "sending_voltage": window_mean(
                signals["sending_node_voltage"], *post_window
            ),
            "receiving_voltage": window_mean(
                signals["receiving_node_voltage"], *post_window
            ),
            "line_current": window_mean(signals["line_current"], *post_window),
        },
    }
)

comparison["pre_relative_error"] = (
    comparison["simulated_pre"] - comparison["analytical_pre"]
).abs() / comparison["analytical_pre"].abs().clip(lower=1e-12)

comparison["post_relative_error"] = (
    comparison["simulated_post"] - comparison["analytical_post"]
).abs() / comparison["analytical_post"].abs().clip(lower=1e-12)

comparison

## 12. Quantify the transient

In [ ]:
def transient_metrics(
    column: str | None,
    final_value: float,
    step_time: float,
    settling_band: float = 0.02,
) -> dict[str, float]:
    if column is None:
        return {
            "minimum": math.nan,
            "maximum": math.nan,
            "overshoot_percent": math.nan,
            "settling_time": math.nan,
        }

    values = df[column].to_numpy(dtype=float)
    post_mask = time >= step_time
    post_time = time[post_mask]
    post_values = values[post_mask]

    minimum = float(np.min(post_values))
    maximum = float(np.max(post_values))

    scale = max(abs(final_value), 1e-12)
    overshoot = (
        max(
            abs(maximum - final_value),
            abs(minimum - final_value),
        )
        / scale
        * 100.0
    )

    lower = final_value * (1.0 - settling_band)
    upper = final_value * (1.0 + settling_band)
    in_band = (post_values >= min(lower, upper)) & (post_values <= max(lower, upper))

    settling_time = math.nan
    for index in range(len(post_values)):
        if in_band[index:].all():
            settling_time = float(post_time[index] - step_time)
            break

    return {
        "minimum": minimum,
        "maximum": maximum,
        "overshoot_percent": overshoot,
        "settling_time": settling_time,
    }


metrics = pd.DataFrame(
    {
        "sending_voltage": transient_metrics(
            signals["sending_node_voltage"],
            final_expected["sending_voltage"],
            STEP_TIME,
        ),
        "receiving_voltage": transient_metrics(
            signals["receiving_node_voltage"],
            final_expected["receiving_voltage"],
            STEP_TIME,
        ),
        "line_current": transient_metrics(
            signals["line_current"],
            final_expected["current"],
            STEP_TIME,
        ),
    }
).T

metrics

## 13. Optional power reconstruction

In [ ]:
power_df = pd.DataFrame({time_column: time})

if signals["source_node_voltage"] is not None and signals["source_current"] is not None:
    power_df["source_power"] = (
        df[signals["source_node_voltage"]] * df[signals["source_current"]]
    )

if (
    signals["receiving_node_voltage"] is not None
    and signals["load_current"] is not None
):
    power_df["load_power"] = (
        df[signals["receiving_node_voltage"]] * df[signals["load_current"]]
    )

if len(power_df.columns) > 1:
    fig, ax = plt.subplots(figsize=(12, 5))
    for column in power_df.columns:
        if column != time_column:
            ax.plot(power_df[time_column], power_df[column], label=column)

    ax.axvline(STEP_TIME, linestyle="--", label=f"source step at {STEP_TIME:g} s")
    ax.set_xlabel("Time [s]")
    ax.set_ylabel("Power [W]")
    ax.set_title("Reconstructed DC powers")
    ax.grid(True)
    ax.legend()
    fig.tight_layout()
    plt.show()

power_df.tail()